# Story Parser

## plan
* take json input, including a story text
* summarize the story scene in markdown
    * story description/summary, genre and scene mood
    * describe the setting, time of day, type of place
    * create a list of characters, their detailed appearance, clothing
* break the story into chunks (paragraph or dialog section)
    * for each chunk, create an image
    * create a sound file


In [1]:
import os
import shutil
import requests
from requests.exceptions import ConnectionError, Timeout, RequestException
import gradio as gr
from typing import List, Dict
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import datetime
import re
import json
from pydantic import BaseModel
from ipyfilechooser import FileChooser
import unicodedata

from PIL import Image
from io import BytesIO
import base64


In [2]:
outputDirMSI="G:\\output\\pythonSD\\"
outputDirDell="G:\\GenerativeAIOutput\\pythonSD"
outputDir=outputDirDell
outputModelDir = ""
promptDir = ""

fc = FileChooser()
fc.default_path = outputDir
fc.title = "<b>Select a story_config.json file</b>"
fc.filter_pattern = '*.json'
display(fc)

FileChooser(path='G:\GenerativeAIOutput\pythonSD', filename='', title='<b>Select a story_config.json file</b>'…

In [215]:
def currentFormattedTime():
    return datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [216]:
class LoraConfig(BaseModel):
    model:str
    strength:float=0.5
    indications:str = ""
    allowedModels: List = []
    blockedModels: List = []

class ImageModelConfig(BaseModel):
    name: str   
    sampler_name: str = "DPM++ 2M Karras" #Euler
    steps: int = 30
    cfg_scale: float = 7


In [ ]:
class StoryConfig(BaseModel):
    llms: list = ["gemma3:4b","gpt-oss:20b"] #qwen3-coder:30b, gemma3:4b, gemma3:12b, gpt-oss:20b
    max_json_generation_attempts: int = 5
    max_json_fix_attempts: int = 5
    repair_llm: str = "gemma3_4b"
    llm_evaluators : list = ["gemma3:4b","gpt-oss:20b"]
    image_models: list[ImageModelConfig] = [
        {"name":"juggernautXL_v8Rundiffusion.safetensors [aeb7e9e689]"},
        {"name":"albedobaseXL_v31Large.safetensors"}
    ]
    auto111_url: str = "http://127.0.0.1"
    ports: List[str] = ["7860","7861"]
    max_images: int = 20
    min_chunk_length: int = 100
    steps: int = 30
    sampler_name: str = "DPM++ 2M Karras" #Euler
    max_story_summary_length:int=500
    max_image_prompt_length:int=350
    positive_prompt: str = ""
    positive_image_prompt: str = ""
    negative_prompt: str = ""
    cfg_scale: float = 7
    seed: int = -1 #-1 for random
    width: int = 1024
    height: int = 1024
    data_file: str = "story.txt"
    LoraKeys: Dict[str, str] = {}
    character_loras : List[LoraConfig] = []
    pose_loras : List[LoraConfig] = []
    style_loras : List[LoraConfig] = []
    environ_loras : List[LoraConfig] = []


In [218]:
# Print the selected path, filename, or both
#print(fc.selected_path)
#print(fc.selected_filename)
#print(fc.selected)

with open(fc.selected, "r") as file:
    raw_config = json.load(file)

config = StoryConfig.model_validate(raw_config)
display(config)


StoryConfig(llms=['mistral-small3.2:latest'], max_json_generation_attempts=5, max_json_fix_attempts=5, repair_llm='gemma3:4b', llm_evaluators=['gpt-oss:20b'], image_models=[ImageModelConfig(name='CHEYENNE_v16.safetensors', sampler_name='DPM++ 2M Karras', steps=30, cfg_scale=7)], auto111_url='http://127.0.0.1', ports=['7860', '7861'], max_images=12, min_chunk_length=100, steps=30, sampler_name='DPM++ 2M Karras', max_story_summary_length=500, max_image_prompt_length=350, positive_prompt='<think>', positive_image_prompt='', negative_prompt='', cfg_scale=7.0, seed=-1, width=1024, height=1024, data_file='story.txt', character_loras=[LoraConfig(model='loras_char\\OldMan.safetensors', strength=0.7, indications='old man, edgar allen poe, male', allowedModels=[], blockedModels=[])], pose_loras=[], style_loras=[], environ_loras=[])

In [219]:
def slugify(value, allow_unicode=False):
    """
    Taken from https://github.com/django/django/blob/master/django/utils/text.py
    Convert to ASCII if 'allow_unicode' is False. Convert spaces or repeated
    dashes to single dashes. Remove characters that aren't alphanumerics,
    underscores, or hyphens. Convert to lowercase. Also strip leading and
    trailing whitespace, dashes, and underscores.
    """
    value = str(value)
    if allow_unicode:
        value = unicodedata.normalize('NFKC', value)
    else:
        value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    value = re.sub(r'[^\w\s-]', '', value.lower())
    return re.sub(r'[-\s]+', '-', value).strip('-_')

In [220]:
def create_output_dir(path):
    display ("Creating output directory at: " + path)
    try:
            os.mkdir(path)
            print(f"Directory '{path}' created successfully.")
            return True
    except FileExistsError:
            print(f"Directory '{path}' already exists.")
            return False
    except Exception as e:
            print(f"An error occurred: {e}") 
            return None   

In [221]:
def copy_file(sourceFile, destDir):
    destFile = os.path.join(destDir, os.path.basename(sourceFile))
    try:
        shutil.copy(sourceFile, destFile)
        print(f"File '{sourceFile}' successfully copied to '{destFile}'")
        return True
    except FileNotFoundError:
        print(f"Error: Source file '{sourceFile}' not found.")
        return False
    except IsADirectoryError:
        print(f"Error: Destination '{destDir}' is a directory, not a file.")
        return False
    except Exception as e:
        print(f"An error occurred: {e}")
        return False

In [222]:
def CreateOutputDirectories(llm_model, checkpoint_image_model):
    _outputModelDir= fc.selected_path + "\\" + slugify(llm_model) #currentFormattedTime()
    create_output_dir(_outputModelDir)

    outputDirSub=_outputModelDir+"\\"+slugify(checkpoint_image_model.name)
    dir_exists = create_output_dir(outputDirSub)
    index=0
    while (dir_exists==False): #dir exists
        index+=1
        outputDirSub= f"{_outputModelDir}\\{slugify(checkpoint_image_model.name)}_{index}"
        dir_exists = create_output_dir(outputDirSub)

    _outputDir = outputDirSub

    copy_file(f"{_outputModelDir}\\story_gallery.json", _outputDir)
    return _outputModelDir, _outputDir 

In [223]:
def CreateOutputDirectoryForCheckpoint(checkpoint_image_model):
    outputDirSub= fc.selected_path +"\\"+slugify(checkpoint_image_model.name)
    dir_exists = create_output_dir(outputDirSub)
    index=0
    while (dir_exists==False): #dir exists
        index+=1
        outputDirSub= f"{fc.selected_path}\\{slugify(checkpoint_image_model.name)}_{index}"
        dir_exists = create_output_dir(outputDirSub)

    _outputDir = outputDirSub

    copy_file(f"{fc.selected_path}\\story_gallery.json", _outputDir)
    return _outputDir 

In [224]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
DeepSeek API Key not set (and this is optional)
Groq API Key exists and begins gsk_
Grok API Key exists and begins xai-
OpenRouter API Key exists and begins sk-


In [225]:
openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [226]:
llama32="llama3.2"
mistralThinker="hf.co/mradermacher/MistralThinker-v1.1-i1-GGUF:Q4_K_M"
qwen3coder30b="qwen3-coder:30b"
gemma3_4b="gemma3:4b"
gemma3_12b="gemma3:12b"
gptOss20b="gpt-oss:20b"

In [227]:
#MODEL=config.llm #gemma3_4b
#REPAIR_MODEL=config.repair_llm #gemma3_4b



In [228]:
system_story_summarizer = f"""You are a helpful assistant that summarizes stories into concise descriptions suitable for background information to generate image prompts. It is very important to keep this summary brief, with no more than {config.max_story_summary_length} characters. This summary will be added to the LLM instructions to the image prompt generator to keep images consistent with one another as a story progresses. Please focus on capturing the key visual elements, settings, characters, and moods of the story in a way that can be effectively translated into image prompts.  Analyze the text below and create Markdown with the following sections:
 1. Story description - a 3-4 sentence summary of the story including genre and scene mood
 2. Story setting: describe the setting including location, type of place, time of day
 3.  create a list of characters, with their detailed appearance, clothing and other visual details. Be specific. It is very important to describe a gender, age, hair color (or bald), hair length and style, and other distinguishing features (e.g. glasses) that should be kept consistent in each image of the story. If the character is not named, give them an appropriate name based on age, gender and location.  Create key features if they are missing from the story (e.g. infer gender or age).  For example, Edgar is a 60 year old male poet, scruffy, ruffled, haggard appearance with balding black and gray hair, and unkempt curly hair, full unkempt beard.  He wears a tweed jacket with patches, and torn brown pants, scuffed dark shoes. 
 4. Key visual elements: highlight any significant objects, colors, or themes that should be included in the image generation. 
 Please format the output in Markdown with appropriate headings for each section. """

In [229]:
system_image_prompt_instruct = """You are a helpful assistant who generates stable diffusion image prompts based on the text from a story.  You will be given a paragraph, stanza or line from a story.  For each paragraph of the story (or stanza of a poem), generate a concise stable diffusion prompt that captures the essence of the paragraph in vivid detail.  Use descriptive language and include artistic styles or techniques where appropriate.  

You will also be given a summary of the overall story to provide context.  Use this to ensure that the prompts you generate are consistent with the story's themes, characters, and settings.
It is very important to maintain consistent character appearances and settings across all prompts.  If a character is described as having specific features (e.g. age, gender, hair color, glasses) or clothing in one paragraph, ensure those details are reflected in all subsequent prompts involving that character. 

It is very important that you output only the image prompt text without any additional commentary or formatting.  The output should be a single, clear prompt suitable for input into a stable diffusion model.
 """



In [230]:
def break_text_into_paragraphs(text):
    """
    Break text into paragraphs by splitting on blank lines.
    Handles various line endings and whitespace variations.
    
    Args:
        text (str): The text to break into paragraphs
        
    Returns:
        list: List of paragraphs (non-empty strings)
    """
    # Split on one or more blank lines (handles different line endings)
    paragraphs = re.split(r'\n\s*\n+', text.strip())
    
    # Remove any leading/trailing whitespace from each paragraph
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    
    return paragraphs

In [231]:
def chunk_paragraphs(paragraphs):
    total_length = sum(len(s) for s in paragraphs)
    chunk_length = config.min_chunk_length
    if (total_length/config.min_chunk_length) > config.max_images:
        chunk_length = total_length/config.max_images
    chunks = []
    chunk = ""
    for p in paragraphs:
        chunk += f"\n{p}"
        if (len(chunk)>=chunk_length):
            chunks.append(chunk)
            chunk = ""
    if len(chunk)>0:
        chunk += f"\n{p}"
    return chunks

In [232]:
def clean_chat_result(text):
    pattern = r"<think>.*?</think>"
    cleaned_text = re.sub(pattern,"",text)
    return cleaned_text

In [233]:
def chat(message, relevant_system_message, llm_model, history = []):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    message = message.encode("ascii", "ignore").decode('ascii') 
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    response = ollama.chat.completions.create(model=llm_model, messages=messages, stream=False)
    if hasattr(response, 'error'):
        print(f"API Error: {response.error}")
        return ""
    if (not hasattr(response, 'choices')):
        print(f"No choices in response to generate image prompt for {paragraph}")
        return ""

    result = response.choices[0].message.content
    

    #display(Markdown(result))
    return result

In [234]:
def GetLorasSubPrompt(lora_array, checkpoint_image_model, prompt, instructions):
    output = ""
    if (not lora_array):
        return output
    for lora in lora_array:
        if (lora.blockedModels):
            if (checkpoint_image_model and (checkpoint_image_model in lora.blockedModels)):
                continue
        if (lora.allowedModels):
            if (checkpoint_image_model and (checkpoint_image_model not in lora.allowedModels)):
                continue
        output += f"Add the lora id '<lora:{lora.model}:{lora.strength}>' to {prompt}'{lora.indications}'\n"
    if (output):
        output = f"{instructions}{output}\n"
    return output

In [235]:
def GetLorasPrompt(checkpoint_image_model):
    output = "" 
    
    output += GetLorasSubPrompt(config.character_loras, checkpoint_image_model,
     " a single unique character who has the features: ","For each of the following character loras, apply the same lora consistently to the same, specific named character in each image.  A single character should have no more than 1 character lora.  A single character lora should not be reused for other, different characters who should have a different appearance:\n")
    output += GetLorasSubPrompt(config.pose_loras, checkpoint_image_model,
    " characters that have the key words or situations: ","Use no more than 1 of the following pose loras for any single image.  Avoid applying duplicate loras that have similar descriptions.  If several loras apply, randomly select between them (avoid always picking the first matching lora).")
    output += GetLorasSubPrompt(config.style_loras, checkpoint_image_model,
    "image style or appearance that has the key words: ","Choose no more than 1 style lora.  If one of the following style loras is applied to one image of a series, it should be consistently applied to all images in that series:\n")
    output += GetLorasSubPrompt(config.environ_loras, checkpoint_image_model,
        " environment apperance, setting or atmosphere that has the key words: ", "Choose no more than 1 environment lora. avoid adding an environment lora, if other loras have already been added (e.g. character or pose loras). if one of the following environmental loras is applied to a setting or environment in one image, that same lora should be applied to all images related to that setting:\n")

    if (output):
        output = f"""Apply no more than a total of 2-3 of the following loras per prompt you generate.  When applying the loras in the prompt, the identifier must follow the form in the following example (starting with '<' and ending with '>'):
        <lora:model:1.0>
        These IDs should be placed as close as possible to the character, object or scene that they describe: {output}"""
    return output

In [236]:
def paragraphToImagePromptWithoutLoras(paragraph, story_summary, llm_model):
    message = f"""{config.positive_prompt}
    Create an image prompt, being concise and efficient, and output  ONLY the prompt (do NOT describe your thinking).  It is very important that the prompt have no more than {config.max_image_prompt_length} characters, and that it capture the scene in the  following paragraph from the story:
    {paragraph}
    

    Here is the summary of the story to provide context:
    {story_summary}
    """
    result = chat(message, system_image_prompt_instruct, llm_model)
    result = clean_chat_result(result)
    return result

In [237]:
def paragraphToImagePrompt(paragraph, story_summary, llm_model, checkpoint_image_model):
    loras = GetLorasPrompt(checkpoint_image_model)
    message = f"""{config.positive_prompt}
    Create an image prompt, being concise and efficient, and output  ONLY the prompt (do NOT describe your thinking).  It is very important that the prompt have no more than {config.max_image_prompt_length} characters, and that it capture the scene in the  following paragraph from the story:
    {paragraph}
    
    {loras}

    Here is the summary of the story to provide context:
    {story_summary}
    """
    result = chat(message, system_image_prompt_instruct, llm_model)
    result = clean_chat_result(result)
    return result

In [238]:
def get_text_file(file_path):
    text = ""
    try:
         with open(file_path, "r", encoding="utf8") as file:
            text = file.read()
    except FileNotFoundError:
        print(f"Error: The file {filename} was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

    return text

In [239]:
def summarize_story_text_file(llm_model):
    story_text = ""
    story_summary = ""
    try:
        filename = fc.selected_path + "\\" + config.data_file
        with open(filename, "r", encoding="utf8") as file:
            story_text = file.read()
            result = chat(f"{config.positive_prompt} {story_text}", system_story_summarizer,llm_model)
            story_summary = clean_chat_result(result)
        print(f"story summary created successfully.")
    except FileNotFoundError:
        print(f"Error: The file {filename} was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

    return story_summary, story_text

In [240]:
class StoryImage(BaseModel):
    storyline: str
    prompt: str
    imagePath: str
    def __init__(self, **data):
        super().__init__(**data)

class StoryImageList(BaseModel):
    summary: str
    paragraphs: List[StoryImage]

In [241]:
def save_json_to_file(json_string, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(json_string)

In [242]:
def Create_Story_ImagePrompts(llm_model,checkpoint_image_model):
    story_summary, story_text = summarize_story_text_file(llm_model)
    paragraphs = break_text_into_paragraphs(story_text)
    chunks = chunk_paragraphs(paragraphs)
    StoryImages = []
    index =0
    for chunk in chunks:
        index += 1
        display(f"Generating image prompt for paragraph {index}/{len(chunks)} \n{chunk}\n")
        imagePrompt = paragraphToImagePrompt(chunk, story_summary, llm_model,checkpoint_image_model)
        StoryImages.append(StoryImage(
            storyline=chunk,
            prompt=imagePrompt,
            imagePath=f"{index:03d}.jpg"
        ))
    storyImageList = StoryImageList(
        summary =story_summary,
        paragraphs=StoryImages
    )
    

    jsonText = storyImageList.model_dump_json(indent=4)

    save_json_to_file(jsonText, outputModelDir + "/story_gallery.json")
    save_json_to_file(jsonText, outputDir + "/story_gallery.json")

    return storyImageList

In [243]:
class Rated_Output(BaseModel):
    model:str
    output:str
    evaluators: Dict[str, float]
    mean_rating:float



class Evaluator(BaseModel):
    model: str
    min_rating: float
    max_rating: float
    mean_rating: float


class LLM_Outputs(BaseModel):
    LLM_Evaluators : List[Evaluator]
    Outputs : List[Rated_Output]
    Text : str

In [244]:
def init_dict_evaluators(evaluators):
    eval_dict = {}
    for index,evaluator in enumerate(evaluators):
        eval_dict[evaluator]=[]
    return eval_dict

In [245]:
def evaluate_content(message, instructions, llm_evaluator):

    result = chat(message, instructions, llm_evaluator)

    return result   

In [246]:
def summarize_evaluators(eval_dict, evaluator_keys):
    output = []
    for key in evaluator_keys:
        vals = eval_dict[key]
        min_rating = max_rating = mean_rating = 0
        if vals:
            min_rating = min(vals)
            max_rating = max(vals)
            mean_rating = sum(vals)/len(vals)
        output.append(Evaluator(
            model=key,
            min_rating=min_rating,
            max_rating=max_rating,
            mean_rating=mean_rating
        ))
    return output

In [247]:
def load_LLM_Outputs_From_File(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        raw_json = json.load(file)
    out_json = LLM_Outputs.model_validate(raw_json)
    return out_json

In [248]:
def getBestLLMOutput(llm_output_json):
    print(llm_output_json)
    max_rated = max(llm_output_json, key=lambda obj: obj.mean_rating)
    return max_rated.output

In [249]:
def Create_Best_Story_Summary():
    llm_Evaluators = config.llm_evaluators
    story_summaries = []
    eval_dict = init_dict_evaluators(llm_Evaluators)
    for index,llm in enumerate(config.llms):
        story_summary, story_text = summarize_story_text_file(llm)
        instructions = f"""You are a helpful evaluator of LLM output.  Several LLMs have generated a summary of a story using the instructions shown below.  Based on how well the LLM followed the instructions and produced a concise and well written summary, Please assign a score between 0 (worst) to 100 (best) for the following summaries.  Do not include any extra text or thinking.  It is very important to respond only with a float or integer number rating that is from 0 to 100. A well stated, concise summary in the length limit starts with a score of 100.  A poorly worded, rambling, poorly organized summary should start with a score at most of 50.  If the model fails to produce any text, then score a 0.  Subtract 20 points for LLM commentary on its thinking or comments embedded in the summary.  Subtract 20 points for a summary that is far longer than the specified length limit.   Subtract 10 points if the individual characters are not well defined.subtract 10 points if the setting is not well described..

        # Here were the original system instructions that the LLM was to follow:
        <original_instructions>
        {system_story_summarizer}
        </original Instructions>

        # This was the original story:
        <original_source_story>
        {story_text}
        </original_source_story>
        """

        message = f""" 
        # Here is the story summary that you should rate from 0 (worst) to 100 (best).  Do not include any extra text or thinking.  It is very important that only with a float or integer number rating that is from 0 to 100:

        <summary>
        {story_summary}
        </summary>
        """
        ratings = []
        model_evals = {}
        for llm_Evaluator in llm_Evaluators:
            result = evaluate_content(message, instructions, llm_Evaluator)
            try:
                rating = float(result)
                if (rating>100.0):
                    rating=100.0
                if (rating<0):
                    rating=0.0
                ratings.append(rating)
                eval_dict[llm_Evaluator].append(rating)
                model_evals[llm_Evaluator] = rating
            except ValueError:
                print(f"summary evaluator {llm_Evaluator} returned an invalid, non-numeric rating ({result}) ")
        mean_rating = None
        if (ratings):
            mean_rating = sum(ratings)/len(ratings)

        summary_eval = Rated_Output(
            model=llm,
            output=story_summary,
            evaluators=model_evals,
            mean_rating=mean_rating
        )
        story_summaries.append(summary_eval)


    evals = summarize_evaluators(eval_dict,llm_Evaluators)
    summaryList = LLM_Outputs(
        LLM_Evaluators =evals,
        Outputs=story_summaries,
        Text = story_text
    )


    jsonText = summaryList.model_dump_json(indent=4)
    
    save_json_to_file(jsonText, fc.selected_path + "/story_summary.json")

    max_rated_summary = max(story_summaries, key=lambda obj: obj.mean_rating)

    return max_rated_summary.output


In [250]:
def Create_Best_Story_ImagePrompt(chunk, story_summary, nPrompt):
    llm_Evaluators = config.llm_evaluators
    prompts = []
    eval_dict = init_dict_evaluators(llm_Evaluators)
    for index,llm in enumerate(config.llms):
        imagePrompt = paragraphToImagePromptWithoutLoras(chunk, story_summary, llm)
   
        instructions = f"""You are a helpful evaluator of LLM output.  Several LLMs have generated a stable diffusion image prompt for a small section of a similar story, poem or other creative text, using the instructions shown below. Please rate the LLM generated prompt based on how well the LLM followed the instructions and produced a concise and well written image prompt. Here are some examples of rating criteria: Is the image prompt consistent with the story summary and the current paragraph?  If the character is named, does the prompt reflect this? Is the prompt too vague, lacking adequate detail? Is there a key character who is described in the paragraph but not in the prompt?  Please assign a score between 0 (worst) to 100 (best) for the following prompt.  Do not include any extra text or thinking.  It is very important that only with a float or integer number rating that is from 0 to 100: 

        # Here were the original system instructions that the LLM was to follow:
        <original_instructions>
        {system_image_prompt_instruct}
        </original Instructions>

        # This is the chunk of story that the LLM generated its image prompt from:
        <original_source_paragraph>
        {chunk}
        </original_source_paragraph>

        # This was the story summary that the LLM recieved as background information:
        <background_story_summary>
        {story_summary}
        </background_story_summary>
        """

        message = f""" 
        # Here is the story summary that you should rate from 0 (worst) to 100 (best).  Do not include any extra text or thinking.  It is very important that only with a float or integer number rating that is from 0 to 100:

        <summary>
        {imagePrompt}
        </summary>
        """
        ratings = []
        model_evals = {}
        for llm_Evaluator in llm_Evaluators:
            result = evaluate_content(message, instructions, llm_Evaluator)
            try:
                rating = float(result)
                if (rating>100):
                    rating=100.0
                if (rating<0):
                    rating=0.0
                ratings.append(rating)
                eval_dict[llm_Evaluator].append(rating)
                model_evals[llm_Evaluator] = rating
            except ValueError:
                print(f"prompt evaluator {llm_Evaluator} returned an invalid, non-numeric rating ({result}) ")
        mean_rating = None
        if (ratings):
            mean_rating = sum(ratings)/len(ratings)

        prompt_eval = Rated_Output(
            model=llm,
            output=imagePrompt,
            evaluators=model_evals,
            mean_rating=mean_rating
        )
        prompts.append(prompt_eval)


    evals = summarize_evaluators(eval_dict,llm_Evaluators)
    promptList = LLM_Outputs(
        LLM_Evaluators =evals,
        Outputs=prompts,
        Text = chunk,
    )


    jsonText = promptList.model_dump_json(indent=4)
    
    save_json_to_file(jsonText, promptDir + f"\\prompt{nPrompt:03d}.json")

    max_rated_prompt = max(prompts, key=lambda obj: obj.mean_rating)

    return max_rated_prompt.output





In [ ]:

def Create_Best_Story_ImagePrompts(story_summary, story_text): #llm_model,checkpoint_image_model
    print(f"Creating image prompts using strory summary: {story_summary}")
    paragraphs = break_text_into_paragraphs(story_text)
    chunks = chunk_paragraphs(paragraphs)
    StoryImages = []
    index =0
    for chunk in chunks:
        index += 1
        display(f"Generating image prompt for paragraph {index}/{len(chunks)} \n{chunk}\n")

        imagePrompt = ""
        file_path = promptDir + f"\\prompt{index:03d}.json"
        if os.path.exists(file_path):
            print(f"'{file_path}' exists. loading...")
            imagePrompt_json = load_LLM_Outputs_From_File(file_path)
            imagePrompt = getBestLLMOutput(imagePrompt_json.Outputs)
        else:
            print(f"'{file_path}' does not exist. Creating a new prompt{index:03d}.json file")
            imagePrompt = Create_Best_Story_ImagePrompt(chunk, story_summary, index)
            
        StoryImages.append(StoryImage(
            storyline=chunk,
            prompt=imagePrompt,
            imagePath=f"{index:03d}.jpg"
        ))
    storyImageList = StoryImageList(
        summary =story_summary,
        paragraphs=StoryImages
    )
    

    jsonText = storyImageList.model_dump_json(indent=4)
    outPath = fc.selected_path + "\\story_gallery.json"
    print(f"writing story_gallery.json to {outPath}")
    save_json_to_file(jsonText,outPath )


    return storyImageList

In [252]:
# following https://civitai.com/articles/4090/make-a-stable-diffusion-easy-interface-with-python
# Import the libraries




In [253]:

#textToImg_url = f"http://127.0.0.1:7860/sdapi/v1/txt2img"

default_negative_prompt = "blurry, low quality, bad anatomy, lowres, error body parts, error hands and fingers, error legs and feet, error face, deformed, blurry, ugly, jpeg artifacts, ugly face, distorted face, extra limbs, mutated hands and fingers, worst quality,"

In [254]:
def find_api_port(host, ports, endpoint = "/login_check/"):
    """
    Attempts to make an API call to a specific endpoint across a list of ports.  Returns the active port or None
    """ 
    for port in ports:
        url = f"{host}:{port}{endpoint}"
        try:
            print(f"Attempting to connect to {url}...")
            # Set a timeout for the request to prevent indefinite waiting
            response = requests.get(url, timeout=10)

            # If successful, process the response and return
            if response.status_code == 200:
                print(f"Success! Connected to port {port}. Status code: {response.status_code}")
                return port
            else:
                print(f"Connected to port {port}, but received status code: {response.status_code}")
        except ConnectionError:
            print(f"Port {port} is closed or service is unreachable.")
        except Timeout:
            print(f"Connection to port {port} timed out.")
        except RequestException as e:
            print(f"An error occurred while connecting to port {port}: {e}")

    print("Failed to connect to any of the specified ports.")
    return None     

In [255]:
auto111_port = find_api_port(
    config.auto111_url, 
    config.ports,
    "/login_check/"
)


Attempting to connect to http://127.0.0.1:7860/login_check/...
Success! Connected to port 7860. Status code: 200


In [256]:
def post_json_to_api(host, port, endpoint, headers, json_data):
    """
    Attempts to make an API call to a specific host, port and endpoint.
    """

    url = f"{host}:{port}{endpoint}"
    try:
        print(f"Attempting to connect to {url}...")
        # Set a timeout for the request to prevent indefinite waiting
        response = requests.post(url, data=json_data, headers=headers)
        #response = requests.get(url, timeout=5)

        # If successful, process the response and return
        if response.status_code == 200:
            print(f"Success! Connected to port {port}. Status code: {response.status_code}")
            return response
        else:
            print(f"Connected to port {port}, but received status code: {response.status_code}")

    except ConnectionError:
        print(f"Port {port} is closed or service is unreachable.")
    except Timeout:
        print(f"Connection to port {port} timed out.")
    except RequestException as e:
        print(f"An error occurred while connecting to port {port}: {e}")

    print("Failed to connect to any of the specified ports.")
    return None

In [257]:
def set_checkpoint(model):
    headers = {'Content-Type': 'application/json'}
    payload = json.dumps({
        "sd_model_checkpoint": model.name
    })
    response = post_json_to_api(
        config.auto111_url,
        auto111_port,
        "/sdapi/v1/options",
        headers,
        payload)
    if (response):
        display(f"succesfully changed checkpoint to {model.name}")
    else:
        display("error changing checkpoint")

In [258]:
# Define the function to call the API
# Must start Automatic 1111 web server before running this code
def call_api(prompt, negative_prompt, filename, checkpoint_image_model):
    # Define the URL of the API endpoint
    steps = checkpoint_image_model.steps
    sampler = checkpoint_image_model.sampler_name
    cfg = checkpoint_image_model.cfg_scale

    data = {
        "prompt": prompt,
        "negative_prompt": default_negative_prompt + negative_prompt,
        "steps": steps if steps else config.steps, #default is 20
        "sampler_name": sampler if sampler else config.sampler_name, # DPM++ 2M Karras, #default is Euler 
        "cfg_scale": cfg if cfg else config.cfg_scale,
        "seed": config.seed, # -1 for random seed
        "width": config.width, #default is 512
        "height": config.height,
        "override_settings": {
            "sd_model_checkpoint": checkpoint_image_model.name #juggernautXL_v8Rundiffusion.safetensors [aeb7e9e689]
        }
    }

    # Convert the data to JSON format
    json_data = json.dumps(data)

    # Set the headers for the request
    headers = {'Content-Type': 'application/json'}

    response = post_json_to_api(
        config.auto111_url,
        auto111_port,
        "/sdapi/v1/txt2img",
        headers,
        json_data)


    # Send the POST request to the API
    #response = requests.post(textToImg_url, data=json_data, headers=headers)
    
    # Check if the request was successful (status code 200)
    if response: #.status_code == 200:
       # Decode the JSON response
        json_response = response.json()

        # Extract the base64 image data from the response
        image_data = json_response.get('images', [''])[0]

        # Decode the base64 image data
        image_bytes = base64.b64decode(image_data)

        # Open the image using PIL
        image = Image.open(BytesIO(image_bytes))

        display(f"Saving image to {filename}")

        #current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        image.save(filename)  # Save the image to a file
        return image
    else:
        if (response):
            print(f"Error: {response.status_code}")
            return None
        else:
             print(f"Error: None was returned from API call")
             return None
        # Return an error message if the request was not successful
        



In [ ]:
def ReplaceLoraKeys(prompt):
    if (not config.LoraKeys):
        return prompt
    result = prompt
    for k in config.LoraKeys.Keys():
        replacement = f"{k} {LoraKeys[k]}"
        result = re.sub(re.escape(k), replacement, result, flags=re.IGNORECASE, count=1)
    return result


In [ ]:
def generate_images_from_story(story_json, checkpoint_image_model, useLoraKeys=False):
    set_checkpoint(checkpoint_image_model)

    for story in story_json:
        story_prompt = story.prompt
        if (useLoraKeys):
            story_prompt=ReplaceLoraKeys(story_prompt)
        prompt = f"""
        {story_prompt} {config.positive_image_prompt}
         """
        
        # This is the story text that will accompany this image:
        # {story.storyline}

        storyline = story.storyline
        negative_prompt = config.negative_prompt
        image = call_api(prompt, negative_prompt,f"{outputDir}\\{story.imagePath}",checkpoint_image_model )
        #display(Markdown(f"{storyline}"))
        #display(image)


In [260]:
def load_story_gallery_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        raw_json = json.load(file)
    out_json = StoryImageList.model_validate(raw_json)
    return out_json

In [261]:
def CreateStoryUsing(llm_model, checkpoint_image_model):
    global outputModelDir, outputDir
    outputModelDir, outputDir = CreateOutputDirectories(llm_model, checkpoint_image_model)

    file_path = outputDir + "/story_gallery.json"
    if os.path.exists(file_path):
        print(f"'{file_path}' exists. loading...")
        storyImages = load_story_gallery_json(file_path)
    else:
        print(f"'{file_path}' does not exist. Creating json file")
        storyImages = Create_Story_ImagePrompts(llm_model,checkpoint_image_model)

    generate_images_from_story(storyImages.paragraphs, checkpoint_image_model)

    copy_file(f".\\slideshow\\index.html", outputDir)


In [262]:
def GetStoryGalleryJsonFromFile():
    file_path =  fc.selected_path + "/story_gallery.json"
    if os.path.exists(file_path):
        print(f"'{file_path}' exists. loading...")
        storyImages = load_story_gallery_json(file_path)
        return storyImages
    else:
        print(f"'{file_path}' does not exist")

In [263]:
def CreateStoryGalleryJsonText():
    global promptDir
    
    storyImages = GetStoryGalleryJsonFromFile()
    if (storyImages):
        return storyImages
    
    print("creating story_gallery.json...")

    summary = ""
    file_path =  fc.selected_path + "/story_summary.json"
    if os.path.exists(file_path):
        print(f"'{file_path}' exists. loading...")
        summary_json = load_LLM_Outputs_From_File(file_path)
        summary = getBestLLMOutput(summary_json.Outputs)
    else:
        print(f"'{file_path}' does not exist. Creating a new story_summary.json file")
        summary = Create_Best_Story_Summary()
    
    story_text_path = fc.selected_path + "\\" + config.data_file
    story_text = get_text_file(story_text_path)

    promptDir= fc.selected_path + "\\" + "_prompts"
    create_output_dir(promptDir)

    storyImageList = Create_Best_Story_ImagePrompts(summary, story_text)
    return storyImageList

In [264]:
def GetCharLoraForParagraph(paragraph_prompt, summary, llm):

    instructions = "" 
    checkpoint_image_model = "" #the character loras will be used regardless of model
    instructions += GetLorasSubPrompt(config.character_loras, checkpoint_image_model,
     " a single unique character who has the features: ","For each of the following character loras, apply the same lora consistently to the same, specifically named character in each image.  A single character should have no more than 1 character lora.  A single character lora should not be reused for other, different characters who should have a different appearance:\n")
    if (instructions):
        instructions = f"""You are a helpful assistant, who adds appropriate and well formatted Lora tags to an image prompt.  You will be given a description of the scene and a summary of the characters in that scene.   Apply no more than a total of 2-3 of the following loras per prompt you generate.  It is very important that When applying the loras in the prompt, the identifier must follow the form in the following example(starting with '<' and ending with '>'): 
        <lora:model:1.0>
        These IDs should be placed as close as possible in the text to the charactere that they describe: {instructions}"""

    message = """
        Add character Lora identifiers to the following paragraph, using information from the summary to identify consistent characters in the text who should have a similar appearance in each image. Add only the stable diffusion Lora identifiers (e.g. <lora:model:1.0>) to the text. Make certain the stable diffusion Lora markup is formatted correctly. Do not modify the text in any other way.  Return only the prompt

        Here is the original prompt to modify:
        <original_prompt>
        {paragraph_prompt}
        </original_prompt>

        Here is a summary of story for background that lists the important characters:
        <summary>
        {summary}
        <summary>
    """
    result = chat(message, instructions,llm_model)
    
    return result

In [265]:
def AddCharLoras(storyImages, llm):
    paragraphs = copy.copy(storyImages.paragraphs)
    for paragraph in storyImages.paragraphs:
        modPrompt = GetCharLoraForParagraph(paragraph.prompt,storyImages.summary, llm)
        paragraph.prompt = modPrompt
    return paragraphs
    



In [ ]:
def CreateStoryImagesFromGalleryFilePrompts():
    global outputDir    

    storyImages = GetStoryGalleryJsonFromFile()
    if (not storyImages):
        storyImages = CreateStoryGalleryJsonText()

    #paragraphsWithCharLoras = AddCharLoras(storyImages, config.repair_llm)
    
    paragraphs = storyImages.paragraphs
 
    for checkpoint_image_model in config.image_models:
        #CreateStoryUsing(llm, checkpoint)

        outputDir = CreateOutputDirectoryForCheckpoint(checkpoint_image_model)
        generate_images_from_story(paragraphs, checkpoint_image_model, True)

        #outputDir = CreateOutputDirectoryForCheckpoint(checkpoint_image_model)
        #generate_images_from_story(paragraphsWithCharLoras, checkpoint_image_model)



        copy_file(f".\\slideshow\\index.html", outputDir)


In [267]:
outputModelDir = fc.selected_path


CreateStoryImagesFromGalleryFilePrompts()


display("DONE...")

'G:\GenerativeAIOutput\pythonSD\stories\poe1/story_gallery.json' exists. loading...


'Creating output directory at: G:\\GenerativeAIOutput\\pythonSD\\stories\\poe1\\cheyenne_v16safetensors'

Directory 'G:\GenerativeAIOutput\pythonSD\stories\poe1\cheyenne_v16safetensors' already exists.


'Creating output directory at: G:\\GenerativeAIOutput\\pythonSD\\stories\\poe1\\cheyenne_v16safetensors_1'

Directory 'G:\GenerativeAIOutput\pythonSD\stories\poe1\cheyenne_v16safetensors_1' created successfully.
File 'G:\GenerativeAIOutput\pythonSD\stories\poe1\story_gallery.json' successfully copied to 'G:\GenerativeAIOutput\pythonSD\stories\poe1\cheyenne_v16safetensors_1\story_gallery.json'
Attempting to connect to http://127.0.0.1:7860/sdapi/v1/options...
Success! Connected to port 7860. Status code: 200


...

'Saving image to G:\\GenerativeAIOutput\\pythonSD\\stories\\poe1\\cheyenne_v16safetensors_1\\010.jpg'

Attempting to connect to http://127.0.0.1:7860/sdapi/v1/txt2img...
Success! Connected to port 7860. Status code: 200


'Saving image to G:\\GenerativeAIOutput\\pythonSD\\stories\\poe1\\cheyenne_v16safetensors_1\\011.jpg'

File '.\slideshow\index.html' successfully copied to 'G:\GenerativeAIOutput\pythonSD\stories\poe1\cheyenne_v16safetensors_1\index.html'


'DONE...'

In [268]:
# summary="# Story Description\nGenre: Gothic Poetry\nThe story describes a man, presumably Edgar, grieving over the lost love of his life, Lenore. As he is reminiscing and reflecting on how much he misses her, a black raven arrives and tells him that she will never come back to him.\n\n# Story Setting\n- Location: Edgar's chambers\n- Type of Place: Interior of a home\n- Time of Day: Midnight\n\n# Characters:\n- **Edgar**: He is a 35 year old grieving man of unspecified gender with messy hair that falls about his forehead.  He wears a black robe with long, wide velvet sleeves, a white cravat around his neck, and black, pointed shoes, and a black, wide brimmed hat.\n- **The Raven**: The Raven is an ancient, ebony bird with a shorn and shaven crest.  He does not move once he perches except to flutter his eyes. He sits perched on a bust of Pallas above a chamber door and speaks only the word \"nevermore\"\n- **Lenore**: A beautiful woman of unspecified gender and age, with long, flowing curls (assumed to be brown).  She is presumably 25 years old.  She dies at an unspecified time before the story and is presumably described as a rare and radiant maiden\n\n# Key Visual Elements\n- Primary Colors: black, purple, white, white, and yellow\n- Important objects: A chamber door, a chamber, a raven, a bust of Pallas, a lamp, and a cushioned seat\n- Themes: death of a lover, grief, hopelessness, and loneliness"